# 05 — Deploy App

Writes `app.yaml` from the widget values, then deploys the Databricks App.

The `app/` folder must already be in the same workspace folder as these notebooks.

Re-run safe — every step is idempotent.

In [ ]:
dbutils.widgets.text("app_name",       "my-first-genie-app", "App Name (lowercase, hyphens only)")
dbutils.widgets.text("warehouse_id",   "",                   "SQL Warehouse ID")
dbutils.widgets.text("genie_space_id", "",                   "Genie Space ID")
dbutils.widgets.text("catalog",        "medtech",            "Catalog")
dbutils.widgets.text("schema",         "sales",              "Schema")

import re, os

app_name       = dbutils.widgets.get("app_name").strip()
warehouse_id   = dbutils.widgets.get("warehouse_id").strip()
genie_space_id = dbutils.widgets.get("genie_space_id").strip()
catalog        = dbutils.widgets.get("catalog").strip()
schema         = dbutils.widgets.get("schema").strip()

if not warehouse_id:
    raise ValueError("warehouse_id is required")
if not genie_space_id:
    raise ValueError("genie_space_id is required — run notebook 04 first")
if not re.match(r'^[a-z][a-z0-9-]*$', app_name):
    raise ValueError("app_name must be lowercase letters, numbers, and hyphens only")

notebook_ws_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir     = os.path.dirname(notebook_ws_path)
app_ws_path      = f"{notebook_dir}/../app"
app_fs_path      = f"/Workspace{app_ws_path}"

print(f"App name:    {app_name}")
print(f"Warehouse:   {warehouse_id}")
print(f"Genie space: {genie_space_id}")
print(f"App path:    {app_fs_path}")

## Step 1 — Write app.yaml

In [ ]:
app_yaml = f"""command:
  - uvicorn
  - app:app
  - --host
  - \"0.0.0.0\"
  - --port
  - \"8000\"

auth:
  - user

env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: \"{warehouse_id}\"
  - name: GENIE_SPACE_ID
    value: \"{genie_space_id}\"

resources:
  - name: sql_warehouse
    sql_warehouse:
      id: {warehouse_id}
      permission: CAN_USE
  - name: genie_space
    genie_space:
      id: {genie_space_id}
      permission: CAN_MANAGE
"""

yaml_path = f"{app_fs_path}/app.yaml"
with open(yaml_path, "w") as f:
    f.write(app_yaml)

print(f"app.yaml written to {yaml_path}")
print(app_yaml)

## Step 2 — Create or Deploy App

In [ ]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

try:
    w.apps.get(name=app_name)
    print(f"App '{app_name}' exists — redeploying.")
except Exception:
    print(f"Creating app '{app_name}' ...")
    w.apps.create(name=app_name, description="J&J MedTech Sales Genie Workshop App")
    print("App created. Waiting for compute to initialize...")
    time.sleep(10)

print(f"Deploying from {app_ws_path} ...")
deployment = w.apps.deploy(
    app_name=app_name,
    source_code_path=app_ws_path,
    mode="SNAPSHOT",
).result()

print(f"Deployment state: {deployment.status.state}")
if str(deployment.status.state) != "DeploymentState.SUCCEEDED":
    raise Exception(f"Deployment failed: {deployment.status.message}")
print("Deployment succeeded.")

## Step 3 — Grant Unity Catalog Permissions

In [ ]:
app_info     = w.apps.get(name=app_name)
sp_client_id = app_info.service_principal_client_id

if sp_client_id:
    print(f"App SP: {sp_client_id}")
    for sql in [
        f"GRANT USE_CATALOG ON CATALOG {catalog} TO `{sp_client_id}`",
        f"GRANT USE_SCHEMA ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`",
        f"GRANT SELECT ON SCHEMA {catalog}.{schema} TO `{sp_client_id}`",
    ]:
        try:
            spark.sql(sql)
            print(f"  OK  {sql}")
        except Exception as e:
            print(f"  ~   {sql}  ({e})")
else:
    print("WARNING: App SP not found. Re-run this cell after the app reaches RUNNING state.")

## Step 4 — Start App and Print URL

In [ ]:
import requests

cfg     = w.config
host    = cfg.host.rstrip("/")
headers = {**dict(cfg.authenticate()), "Content-Type": "application/json"}

requests.post(f"{host}/api/2.0/apps/{app_name}/start", headers=headers)
print("App start requested. Waiting for RUNNING state ...")

app_url = None
for i in range(30):
    time.sleep(10)
    data    = requests.get(f"{host}/api/2.0/apps/{app_name}", headers=headers).json()
    state   = (data.get("app_status") or data.get("status") or {}).get("state", "")
    message = (data.get("app_status") or data.get("status") or {}).get("message", "")
    app_url = data.get("url", "")
    print(f"  [{i+1:02d}/30] {state:<12} {message[:60]}")
    if state == "RUNNING":
        break
    if state in ("CRASHED", "STOPPED"):
        print(f"App {state}. Check logs: Apps > {app_name} > Logs")
        break

if app_url:
    displayHTML(f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
                padding:28px 32px;background:#fff;border:1px solid #e0e0e0;
                border-radius:12px;max-width:600px;margin:16px 0">
      <div style="color:#EB1700;font-size:22px;font-weight:700;margin-bottom:8px">Setup Complete</div>
      <div style="margin-bottom:16px">
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">App URL</span><br>
        <a href="{app_url}" target="_blank" style="font-size:16px;color:#1565C0">{app_url}</a>
      </div>
      <div>
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">Genie Space ID</span><br>
        <code style="font-size:14px;color:#333">{genie_space_id}</code>
      </div>
    </div>
    """)